# 🎙️ Deepfake Voice Detection — Wav2Vec2 Embedding Extraction (Kaggle GPU)

Extracts **768-dim speech embeddings** from ASVspoof 2019 LA using `facebook/wav2vec2-base` on a **T4 GPU**.

### ✅ Before running this notebook:
1. In the right panel → **Accelerator → GPU T4 x1**
2. Attach your uploaded ASVspoof dataset (see Step 1 below for how)
3. Click **Run All**

### 📦 Output files (download these when done):
- `train_embeddings.pt` (~200 MB)
- `dev_embeddings.pt` (~200 MB)

Place them in `c:\Deepfake-voice-detection\data\embeddings\` on your local PC.

---
## Step 1 — How to Upload Your Dataset to Kaggle

> **Do this BEFORE opening this notebook.**

1. Go to [kaggle.com](https://www.kaggle.com) → **Datasets** (top menu) → **New Dataset**
2. Give it a name like `asvspoof2019-la`
3. Upload a **zip file** of your local `LA` folder:
   - On your PC, right-click `C:\Users\rahul\Downloads\AVSspoof_LA\LA` → **Send to → Compressed (zipped) folder**
   - Upload that zip to Kaggle (Train+Dev = ~3 GB, fastest to upload)
4. Click **Create**. Wait for it to finish processing.
5. Come back to this notebook → right panel → **Add data** → search for your dataset → **Add**
6. Note the input path shown — it will be `/kaggle/input/asvspoof2019-la/` (or similar)
7. Update the `DATASET_SLUG` variable in Step 2 below if it differs.

---
## Step 2 — Configuration (Edit This Cell)

In [ ]:
import os
from pathlib import Path

# ─── EDIT THIS if your Kaggle dataset slug is different ───────────────────────
# Check the path shown when you added your dataset in the right panel.
# It will look like:  /kaggle/input/<your-dataset-name>/
DATASET_SLUG = 'asvspoof2019-la'   # <-- change this to match your dataset name
# ─────────────────────────────────────────────────────────────────────────────

# Auto-detect the LA folder inside the dataset
BASE_INPUT = Path(f'/kaggle/input/{DATASET_SLUG}')

# Check common sub-folder patterns
candidates = [
    BASE_INPUT / 'LA',
    BASE_INPUT / 'AVSspoof_LA' / 'LA',
    BASE_INPUT,
]
DATA_ROOT = None
for c in candidates:
    if (c / 'ASVspoof2019_LA_cm_protocols').exists():
        DATA_ROOT = c
        break

if DATA_ROOT is None:
    print('\n❌ Could not auto-detect LA folder. Listing what was found:')
    for p in BASE_INPUT.rglob('ASVspoof2019*'):
        print(' ', p)
    raise RuntimeError(
        f'Dataset not found under {BASE_INPUT}. '
        'Check DATASET_SLUG above matches your Kaggle dataset name.'
    )

OUTPUT_DIR = Path('/kaggle/working/embeddings')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ DATA_ROOT  : {DATA_ROOT}')
print(f'✅ OUTPUT_DIR : {OUTPUT_DIR}')

---
## Step 3 — Verify GPU & Dataset Paths

In [ ]:
import torch

# GPU check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('⚠️  No GPU! Go to right panel → Accelerator → GPU T4 x1, then re-run.')

# Dataset path check
print()
all_ok = True
for sub in [
    'ASVspoof2019_LA_train/flac',
    'ASVspoof2019_LA_dev/flac',
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt',
    'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
]:
    p = DATA_ROOT / sub
    ok = p.exists()
    print(f'  {"✅" if ok else "❌"} {p}')
    if not ok:
        all_ok = False

if all_ok:
    print('\n✅ All paths verified — ready to extract!')
else:
    print('\n❌ Missing paths — check your dataset structure.')

---
## Step 4 — Install & Load Wav2Vec2

In [ ]:
%%capture
!pip install -q transformers torchaudio

In [ ]:
import torchaudio
import transformers
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Model
import time

print(f'torch        {torch.__version__}')
print(f'torchaudio   {torchaudio.__version__}')
print(f'transformers {transformers.__version__}')

MODEL_NAME = 'facebook/wav2vec2-base'
TARGET_SR  = 16_000

print(f'\nLoading {MODEL_NAME} ...')
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
model = Wav2Vec2Model.from_pretrained(MODEL_NAME)
model.eval().to(device)
print(f'✅ Model ready on {device}')

---
## Step 5 — Extraction Function

In [ ]:
SPLIT_CONFIG = {
    'train': {
        'audio_dir' : 'ASVspoof2019_LA_train/flac',
        'protocol'  : 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt',
    },
    'dev': {
        'audio_dir' : 'ASVspoof2019_LA_dev/flac',
        'protocol'  : 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt',
    },
    'eval': {
        'audio_dir' : 'ASVspoof2019_LA_eval/flac',
        'protocol'  : 'ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.eval.trl.txt',
    },
}
LABEL_MAP = {'bonafide': 0, 'spoof': 1}


def parse_protocol(path):
    entries = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                entries.append((parts[1], parts[-1]))  # (filename, label)
    return entries


import soundfile as sf

def load_audio(path):
    try:
        data, sr = sf.read(str(path))
        if data.ndim > 1:
            data = np.mean(data, axis=1)
        wav = torch.from_numpy(data).float().unsqueeze(0)
    except Exception:
        wav, sr = torchaudio.load(str(path))
        if wav.shape[0] > 1:
            wav = wav.mean(0, keepdim=True)
    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)
    return wav.squeeze(0)


def extract_split(split_name, limit=None):
    cfg       = SPLIT_CONFIG[split_name]
    audio_dir = DATA_ROOT / cfg['audio_dir']
    proto     = DATA_ROOT / cfg['protocol']

    entries = parse_protocol(proto)
    if limit:
        entries = entries[:limit]

    total = len(entries)
    print(f'\n[{split_name}] {total:,} clips — starting...', flush=True)

    embeddings, labels, filenames = [], [], []
    skipped = 0
    t0 = time.time()

    with torch.no_grad():
        for i, (fname, label_str) in enumerate(entries):
            flac = audio_dir / f'{fname}.flac'
            if not flac.exists():
                skipped += 1
                continue

            wav    = load_audio(flac)
            inputs = feature_extractor(
                wav.numpy(), sampling_rate=TARGET_SR, return_tensors='pt'
            )
            out = model(inputs.input_values.to(device))
            emb = out.last_hidden_state.mean(1).squeeze(0).cpu()

            embeddings.append(emb)
            labels.append(LABEL_MAP[label_str])
            filenames.append(fname)

            if (i + 1) % 500 == 0 or (i + 1) == total:
                rate = (i + 1 - skipped) / (time.time() - t0)
                eta  = (total - i - 1) / rate / 60
                print(f'  {i+1:>6}/{total}  {rate:.1f} clips/s  ETA ~{eta:.1f} min', flush=True)

    if skipped:
        print(f'  ⚠️  Skipped {skipped} missing files')

    E = torch.stack(embeddings)
    L = torch.tensor(labels, dtype=torch.long)
    print(f'  Embeddings : {E.shape}   bonafide={int((L==0).sum())}  spoof={int((L==1).sum())}')
    return {'embeddings': E, 'labels': L, 'filenames': filenames}


print('✅ Functions defined')

---
## Step 6 — Extract Dev Split (Sanity Check, ~5 min)

In [ ]:
dev_data = extract_split('dev')

dev_out = OUTPUT_DIR / 'dev_embeddings.pt'
torch.save(dev_data, dev_out)
print(f'\n✅ Saved → {dev_out}  ({dev_out.stat().st_size/1e6:.1f} MB)')

---
## Step 7 — Extract Train Split (~15 min)

In [ ]:
train_data = extract_split('train')

train_out = OUTPUT_DIR / 'train_embeddings.pt'
torch.save(train_data, train_out)
print(f'\n✅ Saved → {train_out}  ({train_out.stat().st_size/1e6:.1f} MB)')

---
## Step 8 — (Optional) Extract Eval Split (~40 min)
> Only needed for final benchmark. Skip if you just want to train and test the model first.

In [ ]:
# Uncomment to run eval extraction
# eval_data = extract_split('eval')
# eval_out  = OUTPUT_DIR / 'eval_embeddings.pt'
# torch.save(eval_data, eval_out)
# print(f'\n✅ Saved → {eval_out}  ({eval_out.stat().st_size/1e6:.1f} MB)')
print('Eval extraction skipped (uncomment above to run)')

---
## Step 9 — Summary & Download Instructions

In [ ]:
import os
print('📁 Output files ready for download:')
print('='*55)
for fname in ['train_embeddings.pt', 'dev_embeddings.pt', 'eval_embeddings.pt']:
    fpath = OUTPUT_DIR / fname
    if fpath.exists():
        data  = torch.load(fpath)
        n, d  = data['embeddings'].shape
        mb    = fpath.stat().st_size / 1e6
        print(f'  ✅ {fname:<28} {n:>6} clips  {d}-dim  {mb:.0f} MB')
    else:
        print(f'  ⬜ {fname:<28} (not generated)')
print('='*55)
print()
print('📥 To download:')
print('   Kaggle right panel → Output → Download the .pt files')
print()
print('📂 Place them on your local PC at:')
print('   c:\\Deepfake-voice-detection\\data\\embeddings\\')
print()
print('🎉 Then come back and say "train the model" to continue!')